# 2.2 Calculate degassing path using VolFe

<div style="background-color:#eef8fa; border-left:4px solid #24bdff; padding:10px; border-radius:4px;">
<b> 🧮 &nbsp; VolFe </b> is an open-source Python package for calculating melt-vapor equilibra in the CHOS+ system, including concentration, speciation, and isotope ratios.

More information on VolFe can be found at https://volfe.readthedocs.io/en/latest/

</div>

## 2.2.1 Import packages and note versions

In [ ]:
# Packages that we will use in our code always get imported before we need them.
# This is canonically done at the top of a script.
# ⚠️ Note that this can take a few seconds if it's the first time you're importing these libraries.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import VolFe as vf

import os
os.makedirs("output", exist_ok=True) # creates the output folder

When reporting calculations in manuscripts, it's important to know which version of the Python package the results you are showing used - so we can output those below. This notebook was created using VolFe: 1.0.3.

In [ ]:
print(
    f"\nVolFe: {vf.__version__}"
    )

## 2.2.2 Import data

We'll use the same melt inclusion dataset from notebook <a href="1_1_MI_Temperature_Thermobar.ipynb">1.1 Calculate temperature from MI data using Thermobar</a>, where we've already calculated temperatures.

In [ ]:
# import melt inclusion dataset
MI = pd.read_csv("output/wieser2021_w_temperatures.csv")

# if you haven't run Exercise 1, you can grab the "answer key" file from here:
#results_pvsat_vf = pd.read_csv("files/wieser2021_w_temperatures.csv")

## 2.2.3 Explore options

Again, we'll use the same model options as before, but these can be changed if required - see Section 1.3.3 in <a href="1_3_MI_Pressure_FluidComp_VolFe.ipynb">1.3 Calculate saturation pressures and equilibrium fluid compositions from MI data using VolFe</a> or the VolFe ReadTheDocs for more info.

In [ ]:
# define models
models_vf = [['water','Basalt_Hughes24'],['carbon dioxide','MORB_Dixon95'],['sulfide','ONeill21dil'],['sulfate','ONeill22dil']]
models_vf = vf.make_df_and_add_model_defaults(models_vf)

## 2.2.4 Initial melt composition

We choose the MI with the highest CO<sub>2</sub> content as the starting point for the degassing calcuations

In [ ]:
# row number with highest CO2 content
row = MI['CO2'].idxmax()

print(f"The sample with the highest CO2 content is {row} with {MI['CO2'][row]} wt%")

We also specify the initial <i>f</i>O<sub>2</sub> (ΔFMQ = 0.2 as before)

In [ ]:
DFMQ = 0.2

And combine this all into a single dictionary to use in the calculations in the following sections

In [ ]:
ini_comp = MI.loc[row].to_dict() # convert to dictionary
ini_comp = {k.split(" ")[0]: v for k, v in ini_comp.items()}
ini_comp.update({
    'DFMQ': DFMQ
})

# print initial composition
ini_comp

And create a VolFe sample from this dictionary

In [ ]:
# Define initial melt composition
sample_vf = ini_comp.copy()
sample_vf["CO2ppm"] = sample_vf.pop("CO2_ppm")
sample_vf["STppm"] = sample_vf.pop("S")
sample_vf["Sample"] = sample_vf.pop('Unnamed:')

sample_vf = pd.DataFrame(sample_vf, index=[0])

## 2.2.5 Run calculation

In [ ]:
# run calculation
results_degas_vf = vf.calc_gassing(sample_vf, models = models_vf)

# save your results
results_degas_vf.to_csv("output/results_degas_vf.csv")

# uncomment the line below to interrogate the resulting dataframe
#results_degas_vf

## 2.2.6 Plotting

And we can plot against the pressure and fluid composition <a href="1_3_MI_Pressure_FluidComp_VolFe.ipynb">1.3 Calculate saturation pressures and equilibrium fluid compositions from MI data using VolFe</a>

In [ ]:
results_pvsat_vf = pd.read_csv("output/results_pvsat_vf.csv")

# if you haven't run Exercise 1.2, you can grab the "answer key" file from here:
#results_pvsat_vf = pd.read_csv("files/results_pvsat_vf.csv")

In [ ]:
# if you haven't run this Exercise, you can grab the "answer key" files from here:
#results_degas_vf = pd.read_csv("files/results_degas_vf.csv")

In [ ]:
fig, ((ax1, ax2, ax3),(ax4, ax5, ax6)) = plt.subplots(2, 3, figsize=(12,8)) # create figure with 2 rows and 3 column panels

# melt compositions as circles, fluid compositions as diamonds

# VolFe MI results in purple from Exercise 1
df = results_pvsat_vf
ax1.plot(df['H2OT-eq_wtpc'], df['P_bar'], 'ok', mfc='plum', label = "MI VolFe")
ax2.plot(df['CO2T-eq_ppmw'], df['P_bar'], 'ok', mfc='plum')
ax3.plot(df['xgCO2_mf']*100, df['P_bar'], 'dk', mfc='plum')
ax4.plot(df['ST_ppmw'], df['P_bar'], 'ok', mfc='plum')
ax5.plot(df['Fe3+/FeT'], df['P_bar'], 'ok', mfc="plum")
ax6.plot(100*df['xgSO2_mf']/(df['xgCO2_mf']+df['xgSO2_mf']), df['P_bar'], 'dk', mfc='plum')

# VolFe degassing results from this exercise in purple
df = results_degas_vf
ax1.plot(df['H2OT-eq_wtpc'], df['P_bar'], '-', color='purple', linewidth=2, label = "VolFe")
ax2.plot(df['CO2T-eq_ppmw'], df['P_bar'], '-', color='purple', linewidth=2)
ax3.plot(df['xgCO2_mf']*100, df['P_bar'], '-', color='purple', linewidth=2)
ax4.plot(df['ST_ppmw'], df['P_bar'], '-', color='purple', linewidth=2)
ax5.plot(df['Fe3+/FeT'], df['P_bar'], '-', color='purple', linewidth=2)
ax6.plot(100*df['xgSO2_mf']/(df['xgCO2_mf']+df['xgSO2_mf']), df['P_bar'], '-', color='purple', linewidth=2)

# label axes
ax1.set_ylabel('P (bars)')
ax1.set_xlabel('H$_2$O [m] (wt%)')
ax2.set_xlabel('CO$_2$ [m] (ppm)')
ax3.set_xlabel('CO$_2$ [f] (mol%)')
ax4.set_ylabel('P (bars)')
ax4.set_xlabel('S [m] (ppm)')
ax5.set_xlabel('Fe$^{3+}$/Fe$_T$ [m]')
ax6.set_xlabel('SO$_2$/(SO$_2$+CO$_2$) [f] (mol%)')

ax6.set_xlim(0,10) # axes range in SO2/SO2+CO2 panel

# add legend
ax1.legend(frameon=False)

plt.tight_layout()